# Master Project: AI Insurance Adjuster
## YOLOv8 Training on D-Fire Dataset (Fire & Smoke Detection)

Run this notebook in Google Colab with GPU runtime: `Runtime` -> `Change runtime type` -> `T4 GPU`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Upload master_project.zip to Google Drive, then uncomment:
# !unzip -q /content/drive/MyDrive/master_project.zip -d /content/master_project

# Clone from GitHub:
!git clone https://github.com/Oussama1317/master_project.git /content/master_project

import os
os.makedirs('/content/master_project', exist_ok=True)
%cd /content/master_project

In [ ]:
!pip install -q ultralytics kagglehub

In [ ]:
import kagglehub
import shutil
from pathlib import Path

DATA_DIR = Path("data/raw/dfire")
if not DATA_DIR.exists():
    print("Downloading D-Fire dataset...")
    path = Path(kagglehub.dataset_download("sayedgamal99/smoke-fire-detection-yolo"))
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for item in path.iterdir():
        dest = DATA_DIR / item.name
        if dest.exists():
            shutil.rmtree(str(dest)) if item.is_dir() else dest.unlink()
        shutil.copytree(str(item), str(dest)) if item.is_dir() else shutil.copy2(str(item), str(dest))
    print("Download complete!")
else:
    print("Dataset already exists")

# Verify directories
for split in ['train', 'val', 'test']:
    d = DATA_DIR / 'data' / split / 'images'
    cnt = len(list(d.iterdir())) if d.exists() else 0
    print(f"  {split}: {cnt} images {'✓' if d.exists() else '✗'}")
print("Ready")

In [ ]:
# Create data.yaml and start training
DATA_DIR = Path("data/raw/dfire")
yaml_path = Path("dfire_train.yaml")

train_imgs = len(list((DATA_DIR/'data'/'train'/'images').glob('*')))
val_imgs = len(list((DATA_DIR/'data'/'val'/'images').glob('*')))
test_imgs = len(list((DATA_DIR/'data'/'test'/'images').glob('*')))
print(f"Image counts -> train: {train_imgs}, val: {val_imgs}, test: {test_imgs}")
if train_imgs < 1000:
    raise RuntimeError(f"Dataset incomplete! Only {train_imgs} train images found. Cell 4 download failed.")

# Auto-detect class order from label files (D-Fire filenames encode the category)
import glob as _g
lbl_dir = DATA_DIR / 'data' / 'train' / 'labels'
fire_files = [p for p in _g.glob(str(lbl_dir / '*Fire*.txt'))
              if 'Smoke' not in Path(p).name and 'Neither' not in Path(p).name]
smoke_files = [p for p in _g.glob(str(lbl_dir / '*Smoke*.txt'))
               if 'Fire' not in Path(p).name]

def _first_class(paths):
    for p in paths:
        for line in open(p):
            parts = line.split()
            if parts:
                return int(parts[0])
    return None

fire_idx = _first_class(fire_files)
smoke_idx = _first_class(smoke_files)
if fire_idx is None or smoke_idx is None or fire_idx == smoke_idx or fire_idx not in (0, 1) or smoke_idx not in (0, 1):
    names = ['fire', 'smoke']
    print("Class order not detected reliably, using ['fire', 'smoke']")
else:
    names = ['smoke', 'fire'] if fire_idx == 1 else ['fire', 'smoke']
print(f"Detected class mapping: class {fire_idx} = fire, class {smoke_idx} = smoke -> names = {names}")

yaml_path.write_text(
    f"path: {DATA_DIR.resolve()}\n"
    f"train: data/train/images\n"
    f"val: data/val/images\n"
    f"test: data/test/images\n"
    f"nc: 2\n"
    f"names: {names}\n"
)
print(f"Created {yaml_path.name} with path={DATA_DIR.resolve()}")

from ultralytics import YOLO
model = YOLO("yolov8n.pt")
results = model.train(
    data="dfire_train.yaml",
    epochs=50,
    batch=32,
    imgsz=640,
    patience=15,
    device="cuda",
    workers=4,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    project="runs/train",
    name="dfire_full",
    exist_ok=True,
    pretrained=True,
    verbose=True,
)

In [ ]:
# Validate best model on test set
import glob
from ultralytics import YOLO
import numpy as np

best_pts = sorted(glob.glob("runs/**/best.pt", recursive=True))
print("Found best.pt:", best_pts[-1] if best_pts else "NONE")
best_path = best_pts[-1] if best_pts else "runs/train/dfire_full/weights/best.pt"

val_model = YOLO(best_path)
metrics = val_model.val(data="dfire_train.yaml", split="test", device="cuda")

m = metrics.box
print(f"  mAP50-95: {np.mean(m.map):.4f}")
print(f"  mAP50:    {np.mean(m.map50):.4f}")
print(f"  Precision:{np.mean(m.p):.4f}")
print(f"  Recall:   {np.mean(m.r):.4f}")

In [ ]:
# Save results to Google Drive
import os, glob, shutil
os.makedirs('/content/drive/MyDrive/master_project_results', exist_ok=True)
result_dirs = sorted(glob.glob("runs/detect/runs/train/dfire_full")) + sorted(glob.glob("runs/train/dfire_full"))
if result_dirs:
    src = result_dirs[-1]
    dst = '/content/drive/MyDrive/master_project_results/dfire_full'
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"Saved {src} to Drive")
else:
    print("No results found to save")

In [ ]:
# Download the dfire_full model (not the old cloned one)
import glob, os
from google.colab import files
candidates = glob.glob("runs/**/dfire_full/weights/best.pt", recursive=True)
if not candidates:
    candidates = glob.glob("runs/detect/runs/train/dfire_full/weights/best.pt", recursive=True)
if not candidates:
    raise FileNotFoundError("dfire_full model not found! Did Cell 5 training complete?")
files.download(candidates[-1])